In [9]:
import json
import pathlib
from typing import List, Dict, Any
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

# Initialize LLM once for reuse
llm = ChatOllama(model="qwen3:30b-instruct", temperature=0, reasoning=False, num_ctx=32768)

SYSTEM_PROMPT = """You are a data cleaning specialist for text document chunks.
TASK: Clean and standardize raw text chunks while preserving their core content and meaning.
CLEANING OPERATIONS:
1. Content Normalization:
   - Remove timestamps, dates, and time markers
   - Remove greetings, salutations, small talk and sign-offs
   - Remove metadata (file paths, page numbers, headers, footers)
2. Speaker Label Standardization:
   - Convert all speaker identifiers to simple format:
     * First speaker or questioner → "Speaker A"
     * Second speaker or responder → "Speaker B"
   - Remove complex labels like "说话人:", "[00:05:23] User:", etc.
3. Data Quality:
   - Standardize formatting inconsistencies (quotes, dashes, spacing)
4. Content Preservation Rules:
   - DO NOT remove substantive content or data points
   - DO NOT paraphrase or summarize
   - DO NOT alter technical terms, numbers, or domain-specific language
   - DO maintain document structure and logical flow
OUTPUT FORMAT:
Return ONLY the cleaned text content. No explanations, comments, or status messages.
If the chunk is completely empty or non-substantive after cleaning, return: "[EMPTY_CHUNK]"
"""

def sanitize_content(content: str) -> str:
    """Sanitize a single content string."""
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=content)
    ]
    response = llm.invoke(messages)
    return response.content

def sanitize_batch(contents: List[str], batch_size: int = 10) -> List[str]:
    """Sanitize multiple contents in parallel using LangChain's batch method."""
    # Prepare messages for batch processing
    batch_messages = []
    for content in contents:
        messages = [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=content)
        ]
        batch_messages.append(messages)
    
    # Process in batches for better memory management
    sanitized_contents = []
    for i in range(0, len(batch_messages), batch_size):
        batch = batch_messages[i:i + batch_size]
        print(f"Processing batch {i // batch_size + 1}/{(len(batch_messages) + batch_size - 1) // batch_size} ({len(batch)} items)")
        
        # Use LangChain's batch method for parallel execution
        responses = llm.batch(batch)
        sanitized_contents.extend([response.content for response in responses])
    
    return sanitized_contents


In [10]:
def process_json_file(json_path: pathlib.Path, batch_size: int = 10) -> None:
    """Read, sanitize, and save a single JSON file."""
    print(f"\n{'='*60}")
    print(f"Processing: {json_path.name}")
    print(f"{'='*60}")
    
    # Read JSON file
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    if not isinstance(data, list):
        print(f"Warning: {json_path.name} is not a list, skipping...")
        return
    
    print(f"Found {len(data)} chunks")
    
    # Extract all content
    contents = [chunk.get('content', '') for chunk in data]
    
    # Sanitize all contents in parallel
    print("Sanitizing contents...")
    sanitized_contents = sanitize_batch(contents, batch_size=batch_size)
    
    # Update chunks with sanitized content
    for chunk, sanitized_content in zip(data, sanitized_contents):
        chunk['content'] = sanitized_content
        # Update char_count if it exists
        if 'char_count' in chunk:
            chunk['char_count'] = len(sanitized_content)
    
    # Save back to file
    output_path = json_path  # Overwrite original file
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    
    print(f"✓ Saved sanitized content to {json_path.name}")


In [11]:
def sanitize_all_json_files(outputs_dir: str = "outputs/2025", batch_size: int = 10):
    """Process all JSON files in the outputs directory."""
    # Resolve path - works for both scripts and notebooks
    outputs_path = pathlib.Path(outputs_dir).resolve()
    
    if not outputs_path.exists():
        print(f"Error: Directory {outputs_path} does not exist")
        print(f"Current working directory: {pathlib.Path.cwd()}")
        print(f"Trying relative path from current directory...")
        # Try relative to current working directory
        outputs_path = (pathlib.Path.cwd() / outputs_dir).resolve()
        if not outputs_path.exists():
            print(f"Error: Directory {outputs_path} still does not exist")
            return
    
    # Find all JSON files
    json_files = sorted(outputs_path.glob("*.json"))
    
    if not json_files:
        print(f"No JSON files found in {outputs_path}")
        return
    
    print(f"Found {len(json_files)} JSON files to process")
    print(f"Files: {[f.name for f in json_files]}")
    
    # Process each file
    for json_file in json_files:
        try:
            process_json_file(json_file, batch_size=batch_size)
        except Exception as e:
            print(f"Error processing {json_file.name}: {e}")
            import traceback
            traceback.print_exc()
    
    print(f"\n{'='*60}")
    print("All files processed!")
    print(f"{'='*60}")


In [ ]:
# Run the sanitizer
# Adjust batch_size based on your system's memory and Ollama's capacity
sanitize_all_json_files(outputs_dir="outputs/2025/sanitized", batch_size=10)


Found 10 JSON files to process
Files: ['na_2025_tr_day1A.json', 'na_2025_tr_day1B.json', 'na_2025_tr_day2A.json', 'na_2025_tr_day2B.json', 'na_2025_tr_day3A.json', 'na_2025_tr_day3B.json', 'na_2025_tr_day4A.json', 'na_2025_tr_day4B.json', 'na_2025_tr_day5A.json', 'na_2025_tr_day5B.json']

Processing: na_2025_tr_day1A.json
Found 130 chunks
Sanitizing contents...
Processing batch 1/13 (10 items)
Processing batch 2/13 (10 items)
Processing batch 3/13 (10 items)
Processing batch 4/13 (10 items)
